In [ ]:
from __future__ import annotations

import glob
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks

# =========================================================
# Config
# =========================================================
ROOT_DATA_DIR = Path("D:/mmwave-heart-rate-monitoring-demo/data")
RESULT_DIR = Path("D:/mmwave-heart-rate-monitoring-demo/results") / "alignment"

COND_NAMES = [
    "AWR_steady",
    "AWR_unsteady",
    "IWR_steady",
    "IWR_unsteady",
]

# --- QRS detector params ---
QRS_LO = 8.0
QRS_HI = 20.0
THR_MAD_K = 4.0
REFRACTORY_SEC = 0.30
MWI_SEC = 0.15
REFINE_SEC = 0.10

# --- HR robust params ---
DROP_SHORT_Q = 0.05
WINSOR_P = 0.14
FLOOR_LIKE_MONITOR = True

In [2]:
# =========================================================
# Helpers
# =========================================================
def _extract_sample_name(name: str) -> str | None:
    """
    從檔名中抓出 sample_{n}
    例如：
    sample_1.csv -> sample_1
    sample_1_1lead.csv -> sample_1
    abc_sample_10_test.csv -> sample_10
    """
    m = re.search(r"(sample_\d+)", name, flags=re.IGNORECASE)
    return m.group(1) if m else None


def _estimate_fs_from_time(t: np.ndarray) -> float:
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size < 10:
        return np.nan
    return float(1.0 / np.median(dt))


def _bandpass(x: np.ndarray, fs: float, low: float, high: float, order: int = 2) -> np.ndarray:
    nyq = fs / 2.0
    low = max(0.1, low)
    high = min(high, 0.95 * nyq)
    if not (low < high):
        return x
    b, a = butter(order, [low / nyq, high / nyq], btype="bandpass")
    return filtfilt(b, a, x)


def _highpass(x: np.ndarray, fs: float, cutoff: float = 0.5, order: int = 2) -> np.ndarray:
    nyq = fs / 2.0
    cutoff = min(cutoff, 0.95 * nyq)
    if cutoff <= 0:
        return x
    b, a = butter(order, cutoff / nyq, btype="highpass")
    return filtfilt(b, a, x)


def load_csv_time_signal(csv_path: str | Path) -> tuple[np.ndarray, np.ndarray]:
    data = np.genfromtxt(csv_path, delimiter=",", skip_header=1)

    if data.ndim == 1:
        data = data.reshape(1, -1)

    if data.shape[1] < 2:
        raise RuntimeError(f"CSV 欄位不足（至少需要 time,value）: {csv_path}")

    t = data[:, 0].astype(np.float64)
    x = data[:, 1].astype(np.float64)

    m = np.isfinite(t) & np.isfinite(x)
    t, x = t[m], x[m]

    if t.size < 10:
        raise RuntimeError(f"有效資料太少: {csv_path}")

    return t, x

In [3]:
# =========================================================
# R-peak detection
# =========================================================
def detect_rpeaks_stable(ecg: np.ndarray, fs: float) -> np.ndarray:
    x = ecg.astype(np.float64)
    x -= np.median(x)

    x_hp = _highpass(x, fs, cutoff=0.5, order=2)
    x_qrs = _bandpass(x_hp, fs, low=QRS_LO, high=QRS_HI, order=2)

    dx = np.diff(x_qrs, prepend=x_qrs[0])
    eng = dx * dx

    win = max(3, int(round(MWI_SEC * fs)))
    mwi = np.convolve(eng, np.ones(win) / win, mode="same")

    med = np.median(mwi)
    mad = np.median(np.abs(mwi - med)) + 1e-12
    thr = med + THR_MAD_K * mad

    refractory = int(round(REFRACTORY_SEC * fs))
    prom = 2.0 * mad

    cand, _ = find_peaks(
        mwi,
        height=thr,
        distance=refractory,
        prominence=prom
    )
    if cand.size == 0:
        return np.array([], dtype=int)

    refine_w = int(round(REFINE_SEC * fs))
    rpeaks = []
    for p in cand:
        s = max(0, p - refine_w)
        e = min(len(x_qrs), p + refine_w + 1)
        seg = x_qrs[s:e]
        local = int(np.argmax(np.abs(seg)))
        rpeaks.append(s + local)

    rpeaks = np.array(sorted(set(rpeaks)), dtype=int)

    if rpeaks.size >= 2:
        keep = [rpeaks[0]]
        for idx in rpeaks[1:]:
            if idx - keep[-1] >= refractory:
                keep.append(idx)
        rpeaks = np.array(keep, dtype=int)

    return rpeaks

In [4]:
# =========================================================
# HR estimation
# =========================================================
def _rr_from_peaks(t: np.ndarray, rpeaks: np.ndarray) -> np.ndarray:
    if rpeaks.size < 3:
        return np.array([])

    rr = np.diff(t[rpeaks])
    rr = rr[np.isfinite(rr) & (rr > 0)]
    rr = rr[(rr >= 0.33) & (rr <= 1.50)]  # 40~180 bpm
    return rr


def hr_method_A_drop_short(rr: np.ndarray):
    if rr.size < 2:
        return None
    thr = np.quantile(rr, DROP_SHORT_Q)
    rr2 = rr[rr >= thr]
    if rr2.size < 2:
        rr2 = rr
    return 60.0 / float(np.mean(rr2))


def hr_method_B_winsor(rr: np.ndarray):
    if rr.size < 2:
        return None
    lo = np.quantile(rr, WINSOR_P)
    hi = np.quantile(rr, 1.0 - WINSOR_P)
    rr2 = np.clip(rr, lo, hi)
    return 60.0 / float(np.mean(rr2))


def choose_hr_like_monitor(rr: np.ndarray):
    if rr.size < 2:
        return None, "none"

    cv = float(np.std(rr) / (np.mean(rr) + 1e-12))
    med = float(np.median(rr))
    ratio_max = float(np.max(rr) / (med + 1e-12))

    hrA = hr_method_A_drop_short(rr)
    hrB = hr_method_B_winsor(rr)

    if cv < 0.06:
        return hrA, "drop_short"
    elif (ratio_max > 1.26 and cv < 0.09):
        return hrA, "drop_short"
    elif (ratio_max > 1.25 and cv > 0.12):
        return hrA, "drop_short"
    else:
        return hrB, "winsor"


def hr_display(hr_float: float | None):
    if hr_float is None:
        return None
    if FLOOR_LIKE_MONITOR:
        return float(np.floor(hr_float + 1e-9))
    return float(hr_float)


def estimate_ecg_hr(csv_path: str | Path) -> dict:
    t, x = load_csv_time_signal(csv_path)

    fs = _estimate_fs_from_time(t)
    if not np.isfinite(fs) or fs <= 0:
        fs = 150.0

    t_rel = t - t[0]

    rpeaks = detect_rpeaks_stable(x, fs)
    rr = _rr_from_peaks(t_rel, rpeaks)

    hr_raw, method = choose_hr_like_monitor(rr)
    hr_disp = hr_display(hr_raw)

    cv = float(np.std(rr) / (np.mean(rr) + 1e-12)) if rr.size > 1 else np.nan

    return {
        "fs": float(fs),
        "ECG_HR": np.nan if hr_disp is None else float(hr_disp),
        "hr_method": method,
        "rr_cv": cv,
        "n_rpeaks": int(rpeaks.size),
    }

In [5]:
# =========================================================
# ECG file collection
# =========================================================
def find_ecg_files(cond_name: str) -> list[Path]:
    ecg_root = ROOT_DATA_DIR / cond_name / "raw" / "ECG"
    if not ecg_root.is_dir():
        return []

    all_csv = [Path(p) for p in glob.glob(str(ecg_root / "**" / "*.csv"), recursive=True)]
    csv_list = [p for p in all_csv if re.search(r"sample_\d+", p.name, flags=re.IGNORECASE)]

    csv_list.sort(key=lambda p: (
        int(re.search(r"sample_(\d+)", p.name, flags=re.IGNORECASE).group(1)),
        p.name.lower(),
        str(p).lower(),
    ))
    return csv_list


def build_ecg_hr_map(cond_name: str) -> dict[str, float]:
    """
    直接建立：
    {
        "sample_1": 72.0,
        "sample_2": 68.0,
        ...
    }
    """
    csv_list = find_ecg_files(cond_name)
    hr_map: dict[str, float] = {}

    if not csv_list:
        print(f"⚠️ {cond_name} 找不到 ECG 檔")
        return hr_map

    for csv_path in csv_list:
        sample_name = _extract_sample_name(csv_path.name)
        if sample_name is None:
            continue

        sample_name = sample_name.lower()

        try:
            info = estimate_ecg_hr(csv_path)
            hr_map[sample_name] = info["ECG_HR"]
            print(f"✅ {cond_name} | {sample_name} | ECG_HR={info['ECG_HR']}")
        except Exception as e:
            hr_map[sample_name] = np.nan
            print(f"❌ {cond_name} | {csv_path.name} | {e}")

    return hr_map


# =========================================================
# Merge ECG_HR into alignment CSV
# =========================================================
def update_alignment_csv(cond_name: str):
    in_csv = RESULT_DIR / f"{cond_name}.csv"
    if not in_csv.exists():
        print(f"⚠️ 找不到 alignment csv：{in_csv}")
        return

    df = pd.read_csv(in_csv)
    hr_map = build_ecg_hr_map(cond_name)

    if "sample_name" not in df.columns:
        raise ValueError(f"{in_csv} 缺少欄位 'sample_name'")

    if "ECG_HR" not in df.columns:
        df["ECG_HR"] = np.nan

    # 直接用 sample_name 對應
    sample_name_norm = (
        df["sample_name"]
        .astype(str)
        .str.extract(r"(sample_\d+)", expand=False)
        .str.lower()
    )

    df["ECG_HR"] = sample_name_norm.map(hr_map)

    df.to_csv(in_csv, index=False, encoding="utf-8-sig")
    print(f"💾 已更新：{in_csv}")


# =========================================================
# Main
# =========================================================
def main():
    RESULT_DIR.mkdir(parents=True, exist_ok=True)

    for cond_name in COND_NAMES:
        print(f"\n===== 處理 {cond_name} =====")
        update_alignment_csv(cond_name)

    print("\n全部完成 ✅")


if __name__ == "__main__":
    main()


===== 處理 AWR_steady =====
✅ AWR_steady | sample_0 | ECG_HR=58.0
✅ AWR_steady | sample_1 | ECG_HR=57.0
✅ AWR_steady | sample_2 | ECG_HR=63.0
✅ AWR_steady | sample_3 | ECG_HR=59.0
✅ AWR_steady | sample_4 | ECG_HR=64.0
✅ AWR_steady | sample_5 | ECG_HR=59.0
✅ AWR_steady | sample_6 | ECG_HR=57.0
✅ AWR_steady | sample_7 | ECG_HR=69.0
✅ AWR_steady | sample_8 | ECG_HR=55.0
✅ AWR_steady | sample_9 | ECG_HR=64.0
✅ AWR_steady | sample_10 | ECG_HR=62.0
✅ AWR_steady | sample_11 | ECG_HR=60.0
✅ AWR_steady | sample_12 | ECG_HR=63.0
✅ AWR_steady | sample_13 | ECG_HR=63.0
✅ AWR_steady | sample_14 | ECG_HR=59.0
✅ AWR_steady | sample_15 | ECG_HR=65.0
✅ AWR_steady | sample_16 | ECG_HR=66.0
✅ AWR_steady | sample_17 | ECG_HR=68.0
✅ AWR_steady | sample_18 | ECG_HR=65.0
✅ AWR_steady | sample_19 | ECG_HR=63.0
✅ AWR_steady | sample_20 | ECG_HR=67.0
✅ AWR_steady | sample_21 | ECG_HR=67.0
✅ AWR_steady | sample_22 | ECG_HR=61.0
✅ AWR_steady | sample_23 | ECG_HR=64.0
✅ AWR_steady | sample_24 | ECG_HR=55.0
✅ AWR_st